# Experiment 8: Abstractive Text Summarization Using Multilingual Transformer Models (mT5 and mBART)

**Course**: Computational Linguistics and Natural Language Processing (CLNLP) Lab  
**Student Name**: Satwik Shukla  
**SAP ID**: 500123679  
**Batch**: 1 AIML | B.Tech CSE  

---

### Objectives:
1. Implement and evaluate abstractive text summarization using two multilingual Transformer-based sequence-to-sequence architectures:
   - mT5 (Multilingual Text-to-Text Transfer Transformer; `google/mt5-small`)
   - mBART (Multilingual Bidirectional and Auto-Regressive Transformers; `ai4bharat/IndicBART-XLSum`)
2. Evaluate performance on the Hindi language partition of the benchmark XL-Sum (Cross-Lingual Summarization) dataset.
3. Quantitatively compare model outputs using automatic evaluation metrics:
   - ROUGE-1 (Unigram overlap)
   - ROUGE-2 (Bigram overlap)
   - ROUGE-L (Longest Common Subsequence overlap)
   - mBERT score (Multilingual BERT semantic cosine similarity)


In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import json
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, MBartForConditionalGeneration, AutoModel
from rouge_score import rouge_scorer


## 1. Dataset Loading and Inspection (XL-Sum Hindi)
We load authentic Hindi news articles and human reference summaries from the BBC Hindi split of the XL-Sum benchmark dataset.


In [2]:
samples = []
with open('xlsum_hindi_sample.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            samples.append(json.loads(line.strip()))

dataset_preview = []
for i, item in enumerate(samples[:3]):
    dataset_preview.append({
        'Sample Index': i + 1,
        'Article Title': item['title'],
        'Text Length (chars)': len(item['text']),
        'Summary Length (chars)': len(item['summary'])
    })

df_preview = pd.DataFrame(dataset_preview)
print('=== XL-Sum Hindi Benchmark Dataset Preview ===')
print(df_preview)

test_sample = samples[0]
source_article = test_sample['text'][:600]
reference_summary = test_sample['summary']

print('--- Target Evaluation Sample ---')
print('Article Snippet:')
print(source_article[:250])
print('Human Reference Summary:')
print(reference_summary)


=== XL-Sum Hindi Benchmark Dataset Preview ===
   Sample Index                                      Article Title  \
0             1             साध्वी का ब्रेन मैपिंग और नॉर्को टेस्ट   
1             2       मुशर्रफ़ का पर्चा मंज़ूर, प्रदर्शनों पर लाठी   
2             3  इतनी जल्दी फ़्रांस में ही चमक क्यों खोने लगे र...   

   Text Length (chars)  Summary Length (chars)  
0                 1088                     176  
1                 2370                     257  
2                 4225                     195  
--- Target Evaluation Sample ---
Article Snippet:
नासिक की एक अदालत ने महाराष्ट्र पुलिस के आतंकवाद निरोधक दस्ते (एटीएस) के अनुरोध पर ये आदेश दिया है. सरकारी वकील अजय मिश्रा ने यह जानकारी दी है. कुछ दिन पहले एटीएस ने मध्य प्रदेश के इंदौर शहर से साध्वी प्रज्ञा सिंह ठाकुर को गिरफ़्तार किया था. नासिक की
Human Reference Summary:
मालेगाँव धमाकों के सिलसिले में गिरफ़्तार साध्वी प्रज्ञा सिंह ठाकुर को अब पॉलीग्राफ़, ब्रेन मैपिंग और नॉर्को टेस्ट से गुज़रना होगा. ये सारे टेस्ट झूठ प

## 2. Model 1: Multilingual Text-to-Text Transfer Transformer (mT5)
mT5 (`google/mt5-small`) is an encoder-decoder Transformer pretrained across 101 languages using a unified text-to-text objective.


In [3]:
mt5_model_id = 'google/mt5-small'
mt5_tokenizer = AutoTokenizer.from_pretrained(mt5_model_id)
mt5_model = AutoModelForSeq2SeqLM.from_pretrained(mt5_model_id)

mt5_input_text = 'summarize: ' + source_article
mt5_inputs = mt5_tokenizer(mt5_input_text, return_tensors='pt', truncation=True, max_length=512)

with torch.no_grad():
    mt5_generated_tokens = mt5_model.generate(
        **mt5_inputs,
        max_length=80,
        num_beams=2,
        length_penalty=1.0,
        early_stopping=True
    )

mt5_summary = mt5_tokenizer.decode(mt5_generated_tokens[0], skip_special_tokens=True).strip()

print('=== Model 1: mT5 Generated Summary ===')
print(mt5_summary)


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


=== Model 1: mT5 Generated Summary ===
<extra_id_0> :


## 3. Model 2: Multilingual Bidirectional and Auto-Regressive Transformers (mBART)
mBART is a sequence-to-sequence denoising autoencoder. Here we deploy `ai4bharat/IndicBART-XLSum`, an mBART model fine-tuned on Indic languages for the XL-Sum summarization task.


In [4]:
mbart_model_id = 'ai4bharat/IndicBART-XLSum'
mbart_tokenizer = AutoTokenizer.from_pretrained(mbart_model_id)
mbart_model = MBartForConditionalGeneration.from_pretrained(mbart_model_id)

mbart_inputs = mbart_tokenizer(source_article, return_tensors='pt', truncation=True, max_length=512)

with torch.no_grad():
    mbart_generated_tokens = mbart_model.generate(
        **mbart_inputs,
        max_length=80,
        num_beams=2,
        length_penalty=1.0,
        early_stopping=True
    )

mbart_summary = mbart_tokenizer.decode(mbart_generated_tokens[0], skip_special_tokens=True).strip()

print('=== Model 2: mBART (IndicBART-XLSum) Generated Summary ===')
print(mbart_summary)


Loading weights:   0%|          | 0/267 [00:00<?, ?it/s]

=== Model 2: mBART (IndicBART-XLSum) Generated Summary ===
<s> साध्वी प्रज्ञा सिंह ठाकुर को तीन नवंबर तक पुलिस हिरासत में रखने का फ़ैसला</s>


## 4. Automatic Evaluation Metrics (ROUGE-1, ROUGE-2, ROUGE-L)
We evaluate n-gram overlap and longest common subsequence recall/precision using a multilingual whitespace tokenizer in ROUGE to preserve non-ASCII Devanagari script tokens.


In [5]:
class MultilingualWhitespaceTokenizer:
    def tokenize(self, text):
        return text.strip().split()

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], tokenizer=MultilingualWhitespaceTokenizer())

rouge_scores_mt5 = scorer.score(reference_summary, mt5_summary)
rouge_scores_mbart = scorer.score(reference_summary, mbart_summary)

rouge_comparison = [
    {
        'Model Architecture': 'mT5 (google/mt5-small)',
        'ROUGE-1 F1': round(float(rouge_scores_mt5['rouge1'].fmeasure), 4),
        'ROUGE-2 F1': round(float(rouge_scores_mt5['rouge2'].fmeasure), 4),
        'ROUGE-L F1': round(float(rouge_scores_mt5['rougeL'].fmeasure), 4)
    },
    {
        'Model Architecture': 'mBART (IndicBART-XLSum)',
        'ROUGE-1 F1': round(float(rouge_scores_mbart['rouge1'].fmeasure), 4),
        'ROUGE-2 F1': round(float(rouge_scores_mbart['rouge2'].fmeasure), 4),
        'ROUGE-L F1': round(float(rouge_scores_mbart['rougeL'].fmeasure), 4)
    }
]

df_rouge = pd.DataFrame(rouge_comparison)
print('=== ROUGE Evaluation Results ===')
print(df_rouge)


=== ROUGE Evaluation Results ===
        Model Architecture  ROUGE-1 F1  ROUGE-2 F1  ROUGE-L F1
0   mT5 (google/mt5-small)      0.0000      0.0000      0.0000
1  mBART (IndicBART-XLSum)      0.2609      0.1818      0.2174


## 5. Semantic Similarity Evaluation via Multilingual BERT (mBERT)
We calculate semantic cosine similarity between contextual embeddings of generated summaries and the human reference summary using `google-bert/bert-base-multilingual-cased`.


In [6]:
mbert_model_id = 'google-bert/bert-base-multilingual-cased'
mbert_tokenizer = AutoTokenizer.from_pretrained(mbert_model_id)
mbert_model = AutoModel.from_pretrained(mbert_model_id)

def compute_mbert_similarity(hypothesis_text, reference_text):
    hypo_inputs = mbert_tokenizer(hypothesis_text, return_tensors='pt', truncation=True, max_length=128)
    ref_inputs = mbert_tokenizer(reference_text, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        hypo_emb = mbert_model(**hypo_inputs).last_hidden_state.mean(dim=1)
        ref_emb = mbert_model(**ref_inputs).last_hidden_state.mean(dim=1)
    cosine_sim = F.cosine_similarity(hypo_emb, ref_emb).item()
    return round(float(cosine_sim), 4)

mbert_sim_mt5 = compute_mbert_similarity(mt5_summary, reference_summary)
mbert_sim_mbart = compute_mbert_similarity(mbart_summary, reference_summary)

print('=== mBERT Semantic Cosine Similarity ===')
print('mT5 vs Reference Summary  :', mbert_sim_mt5)
print('mBART vs Reference Summary:', mbert_sim_mbart)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== mBERT Semantic Cosine Similarity ===
mT5 vs Reference Summary  : 0.3134
mBART vs Reference Summary: 0.7926


## 6. Comprehensive Comparative Performance Analysis
We consolidate all evaluation metrics (ROUGE-1, ROUGE-2, ROUGE-L, and mBERT score) into a unified comparative summary table.


In [7]:
full_comparison = [
    {
        'Model Architecture': 'mT5 (google/mt5-small)',
        'ROUGE-1': round(float(rouge_scores_mt5['rouge1'].fmeasure), 4),
        'ROUGE-2': round(float(rouge_scores_mt5['rouge2'].fmeasure), 4),
        'ROUGE-L': round(float(rouge_scores_mt5['rougeL'].fmeasure), 4),
        'mBERT Score': mbert_sim_mt5
    },
    {
        'Model Architecture': 'mBART (ai4bharat/IndicBART-XLSum)',
        'ROUGE-1': round(float(rouge_scores_mbart['rouge1'].fmeasure), 4),
        'ROUGE-2': round(float(rouge_scores_mbart['rouge2'].fmeasure), 4),
        'ROUGE-L': round(float(rouge_scores_mbart['rougeL'].fmeasure), 4),
        'mBERT Score': mbert_sim_mbart
    }
]

df_full_comparison = pd.DataFrame(full_comparison)
print('=== Final Model Comparison (XL-Sum Hindi Summarization) ===')
print(df_full_comparison)


=== Final Model Comparison (XL-Sum Hindi Summarization) ===
                  Model Architecture  ROUGE-1  ROUGE-2  ROUGE-L  mBERT Score
0             mT5 (google/mt5-small)   0.0000   0.0000   0.0000       0.3134
1  mBART (ai4bharat/IndicBART-XLSum)   0.2609   0.1818   0.2174       0.7926
